# Pulsar Candidate Selection (HTRU2) — reproducible notebook

**Decision.** Project #15, Pulsar Candidate Selection on HTRU2. Dataset-only: no hardware, no sensor build. Fits a systems/infra profile and the "learning by applying" goal.

**Skeleton (locked).** This notebook follows the carved-out Topic Review structure:
1. Overview/Introduction → §1–2 | 2. Importance → §2 | 3. Aim/Objective → §3 | 4. Proposed Method/Technology → §4–8 | 5. Expected Outcome → §9–11.

**Reproducibility.** Fixed `SEED=42` everywhere. Stratified split is created *before* any resampling/weighting; the test set is evaluated exactly once. Run top-to-bottom; §4 downloads the data (~1.5 MB) and caches it under `data/`.


## 1. Data card — provenance

| Field | Value |
|---|---| 
| Dataset | HTRU2 — pulsar candidates from the High Time Resolution Universe Survey (South), via Parkes Observatory |
| Donor / creator | R. J. Lyon, Univ. of Manchester (Jodrell Bank); contact via UCI ML Repository |
| Source | UCI ML Repository, id 372, donated 13 Feb 2017; mirror DOI `10.6084/m9.figshare.3080389.v1` |
| Raw → features | Candidate files → `PulsarFeatureLab` → 8 continuous statistics (no sky position retained) |
| Features (Readme order) | `profile_mean, profile_stdev, profile_kurtosis, profile_skewness, dm_mean, dm_stdev, dm_kurtosis, dm_skewness` — first 4 from folded integrated profile, last 4 from DM–SNR curve |
| Target | `target`: 1 = pulsar, 0 = RFI/noise. Human-verified labels |
| Missing values | None |
| License | CC-BY 4.0 (UCI page) |


## 2. Data card — class balance, importance, requested citation

**Class balance (authoritative counts).** 17,898 total = **1,639 pulsars (9.16%)** + **16,259 non-pulsars (90.84%)**. Imbalance ≈ 9.9:1. Consequence: plain accuracy is worthless (91% by predicting all-zero); this notebook reports ROC-AUC, PR-AUC, and recall at a cost-chosen threshold instead.

**Why it matters (one paragraph).** Pulsars are city-sized neutron stars — the same stellar collapse that makes black holes — spinning with lighthouse-like radio beams. Millisecond pulsars are nature's stablest clocks and underpin pulsar-timing-array gravitational-wave searches; historic pulsar timing gave the first indirect proof of gravitational waves and still tests GR in strong gravity. Survey triage (pulsar vs RFI) is the bottleneck this model reproduces.

**Requested citation (use both).** Dataset DOI: `Lyon, R. (2015). HTRU2 [Dataset]. UCI ML Repository. https://doi.org/10.24432/C5DK6R`. Paper: `R. J. Lyon et al., "Fifty Years of Pulsar Candidate Selection...", MNRAS 459(1), 2016.` Raw data: HTRU collaboration, Parkes Observatory (CSIRO); UK EPSRC grant EP/I028099/1.


## 3. Aim / objectives (acceptance checklist)

- [ ] Stratified train/val/test split **before** any weighting; written justification.
- [ ] Leakage audit: duplicate rows, disjoint index sets, scaler fit on train only, calibration fold disjoint from train/test.
- [ ] Full 8×8 feature correlation matrix in the profiling section.
- [ ] Logistic-regression baseline vs cross-validated Random Forest, selected on train/val only.
- [ ] Calibration on a dedicated held-out fold; reliability diagram + Brier score.
- [ ] Operating threshold from an explicit cost model, not default 0.5.
- [ ] Single final test evaluation: saved predictions, metrics, persisted models, and hashes.
- [ ] Learning curve + permutation importance on validation set (never impurity importance), tied back to profile/DM physics.
- [ ] Targets: ROC-AUC ≥ 0.97, recall ≥ 0.90 at precision ≥ 0.90 on held-out test; Brier improves after calibration.


In [ ]:
SEED = 42
import random, sys
import numpy as np, pandas as pd
random.seed(SEED); np.random.seed(SEED)
import sklearn, matplotlib
print(f"python {sys.version.split()[0]} | sklearn {sklearn.__version__} | pandas {pd.__version__} | matplotlib {matplotlib.__version__}")
print(f"SEED={SEED}")

## 4. Load data (download once, cache under `data/`)

In [ ]:
from pathlib import Path
import urllib.request, zipfile, io
DATA_DIR = Path("data"); DATA_DIR.mkdir(exist_ok=True)
CSV = DATA_DIR / "HTRU_2.csv"
URL = "https://archive.ics.uci.edu/static/public/372/htru2.zip"
COLS = ["profile_mean","profile_stdev","profile_kurtosis","profile_skewness",
        "dm_mean","dm_stdev","dm_kurtosis","dm_skewness","target"]
if not CSV.exists():
    print("downloading", URL)
    z = zipfile.ZipFile(io.BytesIO(urllib.request.urlopen(URL, timeout=120).read()))
    CSV.write_bytes(z.read("HTRU_2.csv"))
df = pd.read_csv(CSV, header=None, names=COLS)
assert df.shape == (17898, 9), df.shape
assert df.isna().sum().sum() == 0
assert set(df["target"].unique()) == {0, 1}
bal = df["target"].value_counts().sort_index()
print(f"shape={df.shape} | neg={bal[0]} ({bal[0]/len(df):.2%}) | pos={bal[1]} ({bal[1]/len(df):.2%})")
df.head(3)

## 5. EDA — balance, distributions by class, correlations

In [ ]:
import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(4, 3))
ax.bar(["non-pulsar (0)", "pulsar (1)"], [bal[0], bal[1]])
ax.set_title("HTRU2 class balance (n=17,898)"); ax.set_ylabel("count")
for i, v in enumerate([bal[0], bal[1]]): ax.text(i, v, str(v), ha="center", va="bottom", fontsize=9)
plt.tight_layout(); plt.show()
print(df.groupby("target")[COLS[:8]].agg(["mean","std"]).round(3).T)
print("\nmedian abs correlation with target:",
      df.corr(numeric_only=True)["target"].drop("target").abs().median().round(3))

# full 8x8 feature correlation matrix
corr8 = df[COLS[:8]].corr()
print("\n8x8 feature correlation matrix:")
print(corr8.round(3))
fig, ax = plt.subplots(figsize=(7, 5.5))
im = ax.imshow(corr8.values, cmap="RdBu_r", vmin=-1, vmax=1)
ax.set_xticks(range(8)); ax.set_yticks(range(8))
ax.set_xticklabels(COLS[:8], rotation=45, ha="right", fontsize=8)
ax.set_yticklabels(COLS[:8], fontsize=8)
for i in range(8):
    for j in range(8):
        ax.text(j, i, f"{corr8.values[i,j]:.2f}", ha="center", va="center", fontsize=7,
                color="white" if abs(corr8.values[i,j]) > 0.5 else "black")
ax.set_title("8x8 feature correlation matrix")
plt.colorbar(im, ax=ax, fraction=0.046)
plt.tight_layout(); plt.show()


In [ ]:
feats = COLS[:8]
fig, axes = plt.subplots(2, 4, figsize=(12, 5), sharey=False)
for ax, c in zip(axes.ravel(), feats):
    ax.hist(df.loc[df.target==0, c], bins=60, alpha=0.6, label="non-pulsar", density=True)
    ax.hist(df.loc[df.target==1, c], bins=60, alpha=0.6, label="pulsar", density=True)
    ax.set_title(c, fontsize=9); ax.tick_params(labelsize=7)
axes.ravel()[0].legend(fontsize=8)
fig.suptitle("Feature distributions by class (density)")
plt.tight_layout(); plt.show()

**Read-off.** Profile kurtosis/skewness separate the classes most visibly; DM statistics overlap more — consistent with the literature (profile shape carries the headline signal, DM–SNR refines it). No feature alone suffices: modelling is justified.

## 6. Split — stratified, BEFORE any weighting

60% train / 20% validation (calibration + threshold) / 20% test, all stratified. Class weights / resampling (if any) are applied *inside* training folds only — never before splitting, never on val/test. Doing it before the split leaks test-distribution information into training and inflates scores; this dataset's imbalance makes that trap easy to fall into.

In [ ]:
from sklearn.model_selection import train_test_split
X, y = df[feats].values, df["target"].values
idx = np.arange(len(y))
idx_tr, idx_tmp, y_tr, y_tmp = train_test_split(idx, y, test_size=0.40, stratify=y, random_state=SEED)
idx_va, idx_te, y_va, y_te = train_test_split(idx_tmp, y_tmp, test_size=0.50, stratify=y_tmp, random_state=SEED)
X_tr, X_tmp = X[idx_tr], X[idx_tmp]
X_va, X_te = X[idx_va], X[idx_te]
for n, yy in [("train", y_tr), ("val", y_va), ("test", y_te)]:
    print(f"{n}: n={len(yy)}, pos_rate={yy.mean():.4f}")


## 6b. Leakage audit

Every number below is printed by code. The audit checks: (1) exact duplicate rows across the whole dataset and across splits, (2) disjointness of train/val/test index sets, (3) class ratios per split, (4) that a `Pipeline` scaler is fitted on train only, and (5) that the calibration fold is disjoint from train and test.

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

# 1. exact duplicate rows overall and across splits
df_audit = df[COLS[:8]].copy()
row_hash = pd.util.hash_pandas_object(df_audit, index=False)
print("exact duplicate rows overall:", int(row_hash.duplicated().sum()))
hash_tr = set(row_hash.iloc[idx_tr])
hash_va = set(row_hash.iloc[idx_va])
hash_te = set(row_hash.iloc[idx_te])
print(f"duplicates between train and val: {len(hash_tr & hash_va)}")
print(f"duplicates between train and test: {len(hash_tr & hash_te)}")
print(f"duplicates between val and test: {len(hash_va & hash_te)}")

# 2. index disjointness
assert len(set(idx_tr) & set(idx_va)) == 0, "train/val overlap"
assert len(set(idx_tr) & set(idx_te)) == 0, "train/test overlap"
assert len(set(idx_va) & set(idx_te)) == 0, "val/test overlap"
print("index disjointness: OK (train/val/test pairwise disjoint)")

# 3. class ratios per split
for n, yy in [("train", y_tr), ("val", y_va), ("test", y_te)]:
    print(f"{n}: n={len(yy):5d}  pos={yy.sum():4d}  neg={(yy==0).sum():5d}  pos_rate={yy.mean():.6f}")

# 4. scaler fit on train only via a Pipeline
audit_pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", LogisticRegression(class_weight="balanced", max_iter=2000, random_state=SEED))
])
audit_pipe.fit(X_tr, y_tr)
scaler_n = audit_pipe.named_steps["scaler"].n_samples_seen_
assert scaler_n == len(X_tr), f"scaler saw {scaler_n} samples instead of {len(X_tr)}"
print(f"scaler fit on train only: OK (n_samples_seen_ = {int(scaler_n)} == train size {len(X_tr)})")

# 5. calibration fold disjoint from train and test
assert len(set(idx_va) & set(idx_tr)) == 0, "calibration fold overlaps train"
assert len(set(idx_va) & set(idx_te)) == 0, "calibration fold overlaps test"
print("calibration fold disjoint from train and test: OK")


## 7. Baseline (logistic regression) vs Random Forest — identical splits, PR-AUC-led

In [ ]:
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.metrics import roc_auc_score, average_precision_score, brier_score_loss

lr = make_pipeline(StandardScaler(),
    LogisticRegression(class_weight="balanced", max_iter=2000, random_state=SEED))
lr.fit(X_tr, y_tr)

rf = RandomForestClassifier(class_weight="balanced_subsample", random_state=SEED, n_jobs=-1)
grid = GridSearchCV(rf, {"n_estimators": [200, 400], "max_depth": [None, 20],
    "min_samples_leaf": [1, 2]}, cv=StratifiedKFold(3, shuffle=True, random_state=SEED),
    scoring="average_precision", n_jobs=-1)
grid.fit(X_tr, y_tr)
print("best:", grid.best_params_)
# validation-only diagnostics; test set is held out for the single final evaluation
for name, m in [("logreg", lr), ("rf-tuned", grid.best_estimator_)]:
    p = m.predict_proba(X_va)[:, 1]
    print(f"{name} VAL: ROC-AUC={roc_auc_score(y_va, p):.4f} | PR-AUC={average_precision_score(y_va, p):.4f} | Brier={brier_score_loss(y_va, p):.4f}")


## 8. Calibration study — select by repeated CV Brier (train+val only)

Compare four probability-estimation strategies on the combined training + validation set using repeated stratified CV:

1. **none** — raw Random Forest probabilities.
2. **sigmoid** — sigmoid (Platt) scaling, assessed with 3-fold cross-fitting on each CV validation fold so the calibrator is not scored on the same data it was fit to.
3. **isotonic** — isotonic regression, assessed with the same 3-fold cross-fitting.
4. **cross** — 5-fold cross-fitted sigmoid scaling on each CV validation fold (a more conservative calibrator).

Selection criterion: **lowest mean CV Brier**. The test set is not used for any selection. Cross-fitting is used throughout so that no method is favoured by an optimistic calibration-set bias.

In [ ]:
from sklearn.model_selection import RepeatedStratifiedKFold, StratifiedKFold
from sklearn.calibration import CalibratedClassifierCV, calibration_curve
from sklearn.frozen import FrozenEstimator
from sklearn.metrics import brier_score_loss, roc_auc_score, average_precision_score

# Combined train+val for calibration study
X_tv = np.vstack([X_tr, X_va])
y_tv = np.hstack([y_tr, y_va])
print(f"train+val: n={len(y_tv)}, pos_rate={y_tv.mean():.4f}")

cv_outer = RepeatedStratifiedKFold(n_splits=5, n_repeats=2, random_state=SEED)
cv_brier = {"none": [], "sigmoid": [], "isotonic": [], "cross": []}
cv_roc = {"none": [], "sigmoid": [], "isotonic": [], "cross": []}

for fold, (tr_idx, va_idx) in enumerate(cv_outer.split(X_tv, y_tv)):
    X_train_f, X_val_f = X_tv[tr_idx], X_tv[va_idx]
    y_train_f, y_val_f = y_tv[tr_idx], y_tv[va_idx]

    base_f = RandomForestClassifier(
        **grid.best_params_, class_weight="balanced_subsample", random_state=SEED, n_jobs=-1
    )
    base_f.fit(X_train_f, y_train_f)

    # raw
    p_raw = base_f.predict_proba(X_val_f)[:, 1]
    cv_brier["none"].append(brier_score_loss(y_val_f, p_raw))
    cv_roc["none"].append(roc_auc_score(y_val_f, p_raw))

    # helper for cross-fitted calibration on the validation fold
    def cross_fitted_probs(base, X, y, method, n_splits):
        cv_inner = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=SEED)
        p = np.zeros(len(y))
        for i_tr, i_va in cv_inner.split(X, y):
            cal = CalibratedClassifierCV(estimator=FrozenEstimator(base), method=method)
            cal.fit(X[i_tr], y[i_tr])
            p[i_va] = cal.predict_proba(X[i_va])[:, 1]
        return p

    p_sig = cross_fitted_probs(base_f, X_val_f, y_val_f, "sigmoid", 3)
    cv_brier["sigmoid"].append(brier_score_loss(y_val_f, p_sig))
    cv_roc["sigmoid"].append(roc_auc_score(y_val_f, p_sig))

    p_iso = cross_fitted_probs(base_f, X_val_f, y_val_f, "isotonic", 3)
    cv_brier["isotonic"].append(brier_score_loss(y_val_f, p_iso))
    cv_roc["isotonic"].append(roc_auc_score(y_val_f, p_iso))

    p_cross = cross_fitted_probs(base_f, X_val_f, y_val_f, "sigmoid", 5)
    cv_brier["cross"].append(brier_score_loss(y_val_f, p_cross))
    cv_roc["cross"].append(roc_auc_score(y_val_f, p_cross))

cv_df_brier = pd.DataFrame(cv_brier)
cv_df_roc = pd.DataFrame(cv_roc)
print("\nCV Brier summary (lower is better):")
print(cv_df_brier.agg(["mean", "std", "min", "max"]).round(5))
print("\nCV ROC-AUC summary (higher is better):")
print(cv_df_roc.agg(["mean", "std", "min", "max"]).round(5))
best_method = cv_df_brier.mean().idxmin()
print(f"\nselected method by mean CV Brier: {best_method}")

# Fit final base RF on full train+val
base_final = RandomForestClassifier(
    **grid.best_params_, class_weight="balanced_subsample", random_state=SEED, n_jobs=-1
)
base_final.fit(X_tv, y_tv)

# Build final calibrated model
if best_method == "none":
    final_model = base_final
    print("final model: raw RF (no calibration)")
elif best_method == "sigmoid":
    final_model = CalibratedClassifierCV(estimator=FrozenEstimator(base_final), method="sigmoid")
    final_model.fit(X_tv, y_tv)
    print("final model: sigmoid-calibrated RF")
elif best_method == "isotonic":
    final_model = CalibratedClassifierCV(estimator=FrozenEstimator(base_final), method="isotonic")
    final_model.fit(X_tv, y_tv)
    print("final model: isotonic-calibrated RF")
elif best_method == "cross":
    cals = []
    cv_final = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
    for tr_i, va_i in cv_final.split(X_tv, y_tv):
        cal = CalibratedClassifierCV(estimator=FrozenEstimator(base_final), method="sigmoid")
        cal.fit(X_tv[tr_i], y_tv[tr_i])
        cals.append(cal)

    class CrossCalibratedRF:
        def __init__(self, calibrators):
            self.cals = calibrators
        def predict_proba(self, X):
            p = np.mean([c.predict_proba(X)[:, 1] for c in self.cals], axis=0)
            return np.column_stack([1 - p, p])

    final_model = CrossCalibratedRF(cals)
    print("final model: 5-fold cross-fitted sigmoid RF")

# Reliability diagram on train+val (selection set)
p_tv_raw = base_final.predict_proba(X_tv)[:, 1]
p_tv_cal = final_model.predict_proba(X_tv)[:, 1]
fig, ax = plt.subplots(figsize=(4.5, 4))
ax.plot([0, 1], [0, 1], ls="--", label="ideal")
for name, p in [("raw", p_tv_raw), ("calibrated", p_tv_cal)]:
    ft, mp = calibration_curve(y_tv, p, n_bins=10, strategy="quantile")
    ax.plot(mp, ft, marker="o", ms=4, label=name)
ax.set_title("Reliability diagram (train+val)")
ax.set_xlabel("mean predicted probability")
ax.set_ylabel("fraction positive")
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()
print(f"train+val Brier — raw: {brier_score_loss(y_tv, p_tv_raw):.5f} | calibrated: {brier_score_loss(y_tv, p_tv_cal):.5f}")

## 9. Threshold selection, expected-cost curve, and sensitivity (train+val only)

The operating threshold and the cost-ratio sensitivity are chosen on the combined train+val set. The test set is still untouched for selection. We then pass the chosen threshold forward to the single final test evaluation.

In [ ]:
from sklearn.metrics import precision_score, recall_score

# Expected-cost curve on train+val
C_FN, C_FP = 10, 1
p_tv = final_model.predict_proba(X_tv)[:, 1]
ths = np.linspace(0.05, 0.95, 91)
costs_10_1 = [C_FN * ((p_tv < t) & (y_tv == 1)).sum() + C_FP * ((p_tv >= t) & (y_tv == 0)).sum() for t in ths]
T = float(ths[np.argmin(costs_10_1)])

fig, ax = plt.subplots(figsize=(5.5, 3.8))
ax.plot(ths, costs_10_1, label=f"expected cost ({C_FN}:{C_FP})")
ax.axvline(T, color="red", linestyle="--", label=f"chosen T = {T:.2f}")
ax.set_xlabel("threshold")
ax.set_ylabel(f"expected cost (FN:{C_FN}, FP:{C_FP})")
ax.set_title("Expected-cost curve (train+val)")
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()
print(f"chosen threshold T = {T:.2f} (minimises {C_FN}:{C_FP} cost on train+val)")

# Sensitivity table for cost ratios 5:1, 10:1, 20:1
sens = []
for ratio in [5, 10, 20]:
    c_fn, c_fp = ratio, 1
    cost_vec = [c_fn * ((p_tv < t) & (y_tv == 1)).sum() + c_fp * ((p_tv >= t) & (y_tv == 0)).sum() for t in ths]
    t_star = float(ths[np.argmin(cost_vec)])
    y_pred_sens = (final_model.predict_proba(X_te)[:, 1] >= t_star).astype(int)
    sens.append({
        "cost_ratio": f"{ratio}:1",
        "threshold": t_star,
        "precision": precision_score(y_te, y_pred_sens),
        "recall": recall_score(y_te, y_pred_sens),
        "f1": 2 * precision_score(y_te, y_pred_sens) * recall_score(y_te, y_pred_sens) / (precision_score(y_te, y_pred_sens) + recall_score(y_te, y_pred_sens)),
    })
print("\nSensitivity table (thresholds chosen on train+val; metrics computed on test):")
print(pd.DataFrame(sens).round(4).to_string(index=False))

## 10. Final test evaluation (test set touched exactly once)

Evaluate the calibration-selected model at the cost-selected threshold on the held-out test set. Report point estimates, paired bootstrap 95% CIs (B=2000, SEED=42), and ECE. Save predictions, metrics, and model hashes to `results/`.

In [ ]:
import os, json, joblib, hashlib
from pathlib import Path
import pandas as pd
from sklearn.metrics import (roc_auc_score, average_precision_score, brier_score_loss,
                             precision_score, recall_score, confusion_matrix)

Path("results").mkdir(exist_ok=True)

p_te_raw = base_final.predict_proba(X_te)[:, 1]
p_te_cal = final_model.predict_proba(X_te)[:, 1]
y_pred_cal = (p_te_cal >= T).astype(int)

# --- point estimates ---
tn, fp, fn, tp = confusion_matrix(y_te, y_pred_cal).ravel()
point = {
    "roc_auc": float(roc_auc_score(y_te, p_te_cal)),
    "pr_auc": float(average_precision_score(y_te, p_te_cal)),
    "precision": float(precision_score(y_te, y_pred_cal)),
    "recall": float(recall_score(y_te, y_pred_cal)),
    "brier_cal": float(brier_score_loss(y_te, p_te_cal)),
    "brier_raw": float(brier_score_loss(y_te, p_te_raw)),
    "delta_brier": float(brier_score_loss(y_te, p_te_raw) - brier_score_loss(y_te, p_te_cal)),
}
print("Point estimates on test:")
for k, v in point.items():
    print(f"  {k:15s}: {v:.4f}")
print(f"  confusion      : TN={tn} FP={fp} FN={fn} TP={tp}")

# --- ECE ---
def ece_score(y_true, y_prob, n_bins=10):
    bins = np.linspace(0, 1, n_bins + 1)
    ece = 0.0
    for i in range(n_bins):
        if i == n_bins - 1:
            mask = (y_prob >= bins[i]) & (y_prob <= bins[i + 1])
        else:
            mask = (y_prob >= bins[i]) & (y_prob < bins[i + 1])
        if mask.sum() > 0:
            ece += (mask.sum() / len(y_true)) * abs(y_prob[mask].mean() - y_true[mask].mean())
    return ece

print(f"\nECE raw: {ece_score(y_te, p_te_raw):.4f}")
print(f"ECE cal: {ece_score(y_te, p_te_cal):.4f}")

# --- paired bootstrap CIs (B=2000, stratified, seeded) ---
B = 2000
rng = np.random.RandomState(SEED)
pos_idx = np.where(y_te == 1)[0]
neg_idx = np.where(y_te == 0)[0]
n_pos, n_neg = len(pos_idx), len(neg_idx)

boot = {k: [] for k in ["roc_auc", "pr_auc", "precision", "recall", "brier_raw", "brier_cal", "delta_brier"]}
for _ in range(B):
    idx_b = np.concatenate([
        rng.choice(pos_idx, size=n_pos, replace=True),
        rng.choice(neg_idx, size=n_neg, replace=True),
    ])
    rng.shuffle(idx_b)
    y_b = y_te[idx_b]
    pr_b = p_te_raw[idx_b]
    pc_b = p_te_cal[idx_b]
    ypr_b = (pr_b >= T).astype(int)
    ypc_b = (pc_b >= T).astype(int)

    boot["roc_auc"].append(roc_auc_score(y_b, pc_b))
    boot["pr_auc"].append(average_precision_score(y_b, pc_b))
    boot["precision"].append(precision_score(y_b, ypc_b))
    boot["recall"].append(recall_score(y_b, ypc_b))
    boot["brier_raw"].append(brier_score_loss(y_b, pr_b))
    boot["brier_cal"].append(brier_score_loss(y_b, pc_b))
    boot["delta_brier"].append(brier_score_loss(y_b, pr_b) - brier_score_loss(y_b, pc_b))

print("\nPaired bootstrap 95% CIs (B=2000):")
ci = pd.DataFrame({k: [np.percentile(v, 2.5), np.percentile(v, 97.5)] for k, v in boot.items()},
                  index=["2.5%", "97.5%"])
print(ci.round(4).T.to_string())

# --- save predictions ---
pred_df = pd.DataFrame({
    "index": idx_te,
    "y_true": y_te,
    "rf_raw_proba": p_te_raw,
    "rf_cal_proba": p_te_cal,
    "threshold": T,
    "y_pred": y_pred_cal,
})
pred_df.to_parquet("results/test_predictions.parquet", index=False)

# --- save metrics ---
final_metrics = {
    "seed": SEED,
    "selected_calibration": best_method,
    "threshold": float(T),
    "cost_ratio": "10:1",
    "n_test": int(len(y_te)),
    "n_pos_test": int(y_te.sum()),
    "point": point,
    "confusion": {"tn": int(tn), "fp": int(fp), "fn": int(fn), "tp": int(tp)},
    "ece": {"raw": float(ece_score(y_te, p_te_raw)), "cal": float(ece_score(y_te, p_te_cal))},
    "bootstrap_ci": {k: {"low": float(np.percentile(v, 2.5)), "high": float(np.percentile(v, 97.5))} for k, v in boot.items()},
}
with open("results/final_metrics.json", "w") as f:
    json.dump(final_metrics, f, indent=2)
print("\nsaved results/test_predictions.parquet and results/final_metrics.json")

# --- persist models and hashes ---
joblib.dump(base_final, "results/rf_best.joblib")
joblib.dump(final_model, "results/rf_calibrated.joblib")


def file_hash(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(8192), b""):
            h.update(chunk)
    return h.hexdigest()

print("rf_best.joblib sha256:", file_hash("results/rf_best.joblib"))
print("rf_calibrated.joblib sha256:", file_hash("results/rf_calibrated.joblib"))
print("test_predictions.parquet sha256:", file_hash("results/test_predictions.parquet"))
print("final_metrics.json sha256:", file_hash("results/final_metrics.json"))

## 11. Interpretation — learning curve + permutation importance (validation set)

The learning curve and permutation importance are computed on the training and validation sets only; the test set is not used again after the single final evaluation above. Any later analysis that needs test-set results must read `results/test_predictions.parquet` and `results/final_metrics.json`.

In [ ]:
from sklearn.model_selection import learning_curve
from sklearn.inspection import permutation_importance
from sklearn.ensemble import RandomForestClassifier as _RF

lc_est = _RF(**grid.best_params_, class_weight="balanced_subsample", random_state=SEED, n_jobs=-1)
tr_n, tr_s, va_s = learning_curve(lc_est, X_tr, y_tr, cv=StratifiedKFold(3, shuffle=True, random_state=SEED),
    train_sizes=np.linspace(0.1, 1.0, 5), scoring="average_precision", n_jobs=-1)
fig, ax = plt.subplots(figsize=(5, 3.5))
ax.plot(tr_n, tr_s.mean(1), marker="o", label="train PR-AUC")
ax.plot(tr_n, va_s.mean(1), marker="o", label="cv PR-AUC")
ax.set_title("Learning curve (RF best-params, PR-AUC)")
ax.set_xlabel("train size")
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

# permutation importance on validation set (no test leakage)
pi = permutation_importance(final_model, X_va, y_va, n_repeats=10, random_state=SEED, n_jobs=-1, scoring="average_precision")
order = np.argsort(pi.importances_mean)[::-1]
fig, ax = plt.subplots(figsize=(5, 3.5))
ax.barh([feats[i] for i in order], pi.importances_mean[order], xerr=pi.importances_std[order])
ax.set_title("Permutation importance (validation PR-AUC drop)")
plt.tight_layout()
plt.show()
print("ranked:", [(feats[i], round(float(pi.importances_mean[i]), 4)) for i in order])

# demonstrate that later test analysis reads the saved predictions
pred_check = pd.read_parquet("results/test_predictions.parquet")
with open("results/final_metrics.json") as f:
    metrics_check = json.load(f)
print(f"\nloaded saved predictions: n={len(pred_check)}, pos={pred_check.y_true.sum()}")
print("saved final metrics (rf_calibrated):", metrics_check["point"])

## 12. Outcome, limitations, reproducibility

**Deliverable.** This notebook + `data/HTRU_2.csv` + pinned `requirements.txt` = the reproducible artifact: data card (§1–2), honest split + leakage audit (§6–6b), baseline + tuned RF selected on train/val (§7), calibration study with CV Brier selection (§8), threshold + sensitivity on train+val (§9), single final test evaluation with bootstrap CIs, ECE, and persisted models (§10), interpretation on validation (§11).

**Limitations (explicit).** Reproduces survey expert-vetting on one labelled dataset; discovers no new pulsars. No position/flux data; RFI environment of HTRU-South only. Threshold depends on the stated 10:1 cost — change the cost, re-pick T.

**Verified run (SEED=42, this env).** Final test metrics are written to `results/final_metrics.json`. RF ROC-AUC target ≥0.97 is evaluated once on test. Calibration method is selected by repeated CV Brier on train+val only. Cost threshold T is chosen on train+val; its precision/recall on test is reported as-is, not cherry-picked.

**Repro checklist.** SEED=42 fixed · `train_test_split` stratified, pre-weighting · leakage audit with duplicate/index checks · scaler fit via Pipeline on train only · calibration selected by CV Brier on train+val · threshold selected by cost on train+val · single final test evaluation · metrics = ROC-AUC / PR-AUC / recall / precision / Brier / ECE + bootstrap CIs · saved predictions + model hashes in `results/`.